# Stellar population fit by ppxf — E-INSPIRE

## Import relevant modules

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits
import os
from os.path import basename
from copy import copy

from ppxf.ppxf import ppxf
import ppxf.ppxf_util as util
import ppxf.sps_util as lib

from src.ned_calculator import NedCalculator
from src.der_snr import DER_SNR
import os

## Create necessary functions

In [2]:
def bootstrap_residuals(model, resid, wild=True):
    """
    https://en.wikipedia.org/wiki/Bootstrapping_(statistics)#Resampling_residuals
    https://en.wikipedia.org/wiki/Bootstrapping_(statistics)#Wild_bootstrap

    Davidson & Flachaire (2008) eq.(12) gives the recommended form
    of the wild bootstrapping probability used here.

    https://doi.org/10.1016/j.jeconom.2008.08.003

    :param spec: model (e.g. best fitting spectrum)
    :param res: residuals (best_fit - observed)
    :param wild: use wild bootstrap to allow for variable errors
    :return: new model with bootstrapped residuals

    """
    if wild:    # Wild Bootstrapping: generates -resid or resid with prob=1/2
        eps = resid*(2*np.random.randint(2, size=resid.size) - 1)
    else:       # Standard Bootstrapping: random selection with repetition
        eps = np.random.choice(resid, size=resid.size)

    return model + eps

In [3]:
def read_fits_summary(fitsfile):
    
    hdu = fits.open(fitsfile)
    
    age_grid = hdu['age_grid'].data
    weights = hdu['pp_weights'].data.reshape(hdu['reg_dim'].data)
    
    name = hdu[0].header['name']
    z = hdu[0].header['z']
    hdu.close()
    del hdu
    
    wei1 = weights.sum(axis=1)
    wei1/=wei1.sum()
    
    wei1_rev = copy(wei1[::-1])
    
    ages = age_grid[:,0]
    ages1 = (ages[-1]-ages)[::-1]+(ages[1]-ages[0])
    
    agesplot = np.concatenate([np.array([0.]),ages1])
    weiplot = np.concatenate([np.array([0.]),np.cumsum(wei1_rev)])
    
    agesplot = np.concatenate([agesplot,np.array([agesplot[-1]+(agesplot[-1]-agesplot[-2])])])
    weiplot = np.concatenate([weiplot,np.array([weiplot[-1]])])
    
    nedcalc = NedCalculator(z)
    univ_age = nedcalc.zage_Gyr
    
    agesplot = [a if a<univ_age else univ_age for a in agesplot]
    
    return name,z,agesplot,weiplot,univ_age

In [4]:

def plot_sfh(ax,fitsfile,col_line,legend_on=False):
    
    name,z,agesplot,weiplot,univ_age = read_fits_summary(fitsfile)

    ax.set_title(name,fontsize=18,weight='bold')
    ax.set_xlim(0,13.5)
    ax.set_ylim(-0.05,1.05)
    ax.set_xlabel('Time since BB (Gyr)',fontsize=15)
    ax.set_ylabel('Cumulative mass %',fontsize=15)
    ax.minorticks_on()
    ax.tick_params(axis='both',which='both',direction='in',labelsize=15)
    
    ax.axhline(0.75,color='gray',alpha=0.2)
    ax.text(12,0.76,'75%',style='italic',color='gray')
    ax.axhline(0.95,color='gray',alpha=0.2)
    ax.text(12,0.96,'95%',style='italic',color='gray')
    ax.axvline(3.,color='gray',alpha=0.2,linestyle='-.',linewidth=2.)
    ax.axvline(univ_age,color='gray',linestyle='-.',linewidth=2.)
    
    ax.plot(agesplot,weiplot,color=col_line,linewidth=3.)
    
    ax.text(univ_age-0.1,0.,'today',color='gray',style='italic',rotation=90,horizontalalignment='right')
    ax.text(2.9,0.,'z~2',style='italic',color='gray',rotation=90,horizontalalignment='right')
    ax.set_xticks([0.,2.,4.,6.,8.,10.,12.])
    
    if legend_on==True:
        ax.legend(frameon=False,bbox_to_anchor=(univ_age/ax.get_xlim()[1], -0.05),loc='lower right',prop={'size':12,'weight':'bold'},labelspacing=0.1,labelcolor='linecolor')
        
    return agesplot,weiplot

In [5]:
def line2p(p1,p2,x):
    
    x1,y1 = p1
    x2,y2 = p2
    
    m = (y2-y1)/(x2-x1)
    q = -(y2-y1)/(x2-x1)*x1+y1
    
    return m*x+q

def line2p_rev(p1,p2,y):
    
    x1,y1 = p1
    x2,y2 = p2
    
    return (y-y1)/(y2-y1)*(x2-x1)+x1 if y2!=y1 else x2
    
def get_values_from_sfh(univ_age,sfh_table,ycol):
    '''
    INPUTS:
        univ_age: the age of the universe at the redshift of the galaxy in Gyr
        sfh_table: the plotted sfh in the form of a pandas DataFrame. It needs to have a column named "time"
        ycol: is the name of the column of the sfh_table we want to use in order to retrieve the different values we want to compute
    
    OUTPUTS:
        y_z2: mass formed at redshift~2
        x_075: time to form 75% of the mass (t_75)
        x_090: time to form 90% of the mass (t_90)
        x_100: time to form 100% of the mass (t_fin)
        dor_90: Degree of Relicness using x_090
        dor_100: : Degree of Relicness using x_100
        
    '''
    
    tt = []
    tt90 = []
    tt100 = []
    
    ii=0
    while sfh_table.iloc[ii][ycol]<0.75:
        tt.append((ii,sfh_table.iloc[ii][ycol]))
        ii+=1
    tt.append((ii,sfh_table.iloc[ii][ycol]))
    
    yy = sfh_table.iloc[np.array(tt[-2:])[:,0]]
    
    xx = line2p_rev(yy[['time',ycol]].iloc[0].values,yy[['time',ycol]].iloc[1].values,0.75)
    
    ii=0
    while sfh_table.iloc[ii][ycol]<0.9 :
        tt90.append((ii,sfh_table.iloc[ii][ycol]))
        ii+=1
    tt90.append((ii,sfh_table.iloc[ii][ycol]))
    
    yy90 = sfh_table.iloc[np.array(tt90[-2:])[:,0]]
    
    xx90 =  line2p_rev(yy90[['time',ycol]].iloc[0].values,yy90[['time',ycol]].iloc[1].values,0.9)
    
    ii=0
    while sfh_table.iloc[ii][ycol]<0.998 :
        tt100.append((ii,sfh_table.iloc[ii][ycol]))
        ii+=1
    tt100.append((ii,sfh_table.iloc[ii][ycol]))
    
    yy100 = sfh_table.iloc[np.array(tt100[-2:])[:,0]]
    
    xx100 =  line2p_rev(yy100[['time',ycol]].iloc[0].values,yy100[['time',ycol]].iloc[1].values,0.998)
    
    tt_rev = [(0,0)]
    
    for i in range(1,len(sfh_table['time'])):
        
        p1 = sfh_table['time'].iloc[i-1],sfh_table[ycol].iloc[i-1]
        p2 = sfh_table['time'].iloc[i],sfh_table[ycol].iloc[i]
        
        xs = np.arange(sfh_table['time'].iloc[i-1],sfh_table['time'].iloc[i]+0.1,0.1)
        ys = line2p(p1,p2,xs)

                
        for x,y in zip(xs,ys):
            tt_rev.append((round(x,2),y))


    tt_rev = np.array(tt_rev)

    # this is the mass formed at redshift ~2
    y_z2 = round(tt_rev[:,1][np.where(tt_rev[:,0]==2.90)[0]][0],5)
    
    # these are the times at 75%, 90%, and 100% formed mass
    x_075 = round(xx,5)
    x_090 = round(xx90,5)
    x_100 = round(xx100,5)
    
    dor_90 = (y_z2+0.5/x_075+(0.7+(univ_age-x_090)/univ_age))/3
    dor_100 = (y_z2+0.5/x_075+(0.7+(univ_age-x_100)/univ_age))/3
    
    
    
    return y_z2,x_075,x_090,x_100,dor_90,dor_100

## Load in the shortlist csv

In [6]:
from pathlib import Path
DATA_DIR = Path('../data') if Path('../data').exists() else Path('data')

# Column mapping: rename E-INSPIRE master catalogue columns to match INSPIRE names used throughout this notebook
einspire_to_inspire = {
    'GALAXY ID':        'ID_INSPIRE',
    'velDisp_ppxf_res': 'Vdisp_XSH',
    'MgFe':             'AlphaFe',
    'z':                'zspec_XSH',
}

df_in = pd.read_csv(DATA_DIR / 'E-INSPIRE_I_master_catalogue.csv')
df_in = df_in.rename(columns=einspire_to_inspire)
#df_in = df_in.head(10)
df_in.head()

,ID_INSPIRE,ra,dec,plate,mjd,fiberid,objid,deVRad_r,deVRadErr_r,expRad_r,...,t_90_unr,t_90_reg,t_90_alphaplus,t_90_alphamin,t_100,t_100_unr,t_100_reg,t_100_alphaplus,t_100_alphamin,DoR
0,J2350+0609,357.704920,6.158195,4405,55854,591,1.240000e+18,0.960140,0.017636,0.924479,...,1.35128,1.39994,NaN,1.41404,10.84242,2.38420,10.84242,NaN,2.45802,0.564630
1,J1647+2956,251.955552,29.949596,1342,52793,503,1.240000e+18,0.409805,0.016891,0.548107,...,1.74192,0.78766,NaN,1.68151,10.73174,4.52743,7.34051,NaN,10.73174,0.529239
2,J1437+1800,219.335100,18.007743,2775,54535,128,1.240000e+18,0.313001,0.010724,0.418768,...,1.91359,2.48573,NaN,5.20643,10.59350,4.74365,10.59350,NaN,9.95189,0.436232
3,J0921+6020,140.340674,60.334325,485,51909,432,1.240000e+18,1.008870,0.017163,0.797932,...,1.72799,3.10466,NaN,1.56114,11.23258,4.59661,10.91191,NaN,11.23258,0.486632
4,J0807+5317,121.808009,53.294277,1781,53297,341,1.240000e+18,0.726614,0.013709,0.818694,...,1.11047,2.75532,NaN,1.38459,3.59529,2.05288,3.59529,NaN,2.58137,0.761979


## Iterate over each row to fit the stellar population of each object

In [7]:
nrand = 9

In [8]:
"""from astropy.io import fits

# Replace with your actual FITS file path
file_path = "../data/INSPIRE_SPEC/J0211-3155_merged_gdago_LOG.fits"


with fits.open(file_path) as hdul:
    hdul.info()
    
    # For each HDU, print detailed information about its structure
    for i, hdu in enumerate(hdul):
        print(f"\n--- HDU {i} ({hdu.name}) ---")
        if hdu.header:
            print("Header keywords:")
            for key in hdu.header:
                print(f"  {key} = {hdu.header[key]}")
        
        if hasattr(hdu, 'data') and hdu.data is not None:
            if hasattr(hdu.data, 'dtype') and hasattr(hdu.data, 'shape'):
                print(f"Data shape: {hdu.data.shape}")
                print(f"Data type: {hdu.data.dtype}")
                
                # If this is a table, show column info
                if hasattr(hdu.data, 'columns'):
                    print("Columns:")
                    for col in hdu.data.columns:
                        print(f"  {col.name}: {col.format}")"""
print("For investigating how a fits file is")

For investigating how a fits file is


In [ ]:
import importlib, src.einspire_refit
importlib.reload(src.einspire_refit)
from src.einspire_refit import fit_galaxy
from concurrent.futures import ProcessPoolExecutor, as_completed

os.makedirs('einspire_refitting_results', exist_ok=True)

rows = df_in.to_dict('records')
for r in rows:
    r['_nrand'] = nrand

results = []
with ProcessPoolExecutor(max_workers=8) as executor:
    futures = {executor.submit(fit_galaxy, r): r['ID_INSPIRE'] for r in rows}
    for future in as_completed(futures):
        gal = futures[future]
        try:
            results.append(future.result())
        except Exception as e:
            print(f'FAILED {gal}: {e}')
            results.append({'ID_INSPIRE': gal, 'error': str(e)})

print(f'\nCompleted {sum(r["error"] is None for r in results)}/{len(rows)} galaxies successfully')

Emission lines included in gas templates:
['Balmer' '[OII]3726_d1' '[OII]3726_d2' '[NeIII]3968' '[NeIII]3869'
 'HeII4687' 'HeI5876' '[OIII]5007_d' '[OI]6300_d']
Emission lines included in gas templates:
['Balmer' '[OII]3726_d1' '[OII]3726_d2' '[NeIII]3968' '[NeIII]3869'
 'HeII4687' 'HeI5876' '[OIII]5007_d' '[OI]6300_d']
Emission lines included in gas templates:
['Balmer' '[OII]3726_d1' '[OII]3726_d2' '[NeIII]3968' '[NeIII]3869'
 'HeII4687' 'HeI5876' '[OIII]5007_d' '[OI]6300_d']
Emission lines included in gas templates:
['Balmer' '[OII]3726_d1' '[OII]3726_d2' '[NeIII]3968' '[NeIII]3869'
 'HeII4687' 'HeI5876' '[OIII]5007_d' '[OI]6300_d']
Emission lines included in gas templates:
['Balmer' '[OII]3726_d1' '[OII]3726_d2' '[NeIII]3968' '[NeIII]3869'
 'HeII4687' 'HeI5876' '[OIII]5007_d' '[OI]6300_d']
Emission lines included in gas templates:
['Balmer' '[OII]3726_d1' '[OII]3726_d2' '[NeIII]3968' '[NeIII]3869'
 'HeII4687' 'HeI5876' '[OIII]5007_d' '[OI]6300_d']
Emission lines included in gas tem

In [ ]:
# Inspect a result
results[0]

{'ID_INSPIRE': 'J1317+4336',
 'logAge': 10.022639027201166,
 '[M/H]': 0.3316749137086642,
 'SNR': 14.840348199323197,
 'mass_frac': 0.99983,
 'time_75': 1.00836,
 'time_90': 1.85064,
 'time_100': 2.4873,
 'dor_100': 0.7787062012250239,
 'univ_age': 11.201004124775709,
 'mass_frac_reg': 0.99983,
 'time_75_reg': 0.47793,
 'time_90_reg': 1.85064,
 'time_100_reg': 2.4873,
 'dor_100_reg': 1.174649287365126,
 'mass_frac_unr': 1.0,
 'time_75_unr': 1.00836,
 'time_90_unr': 1.43734,
 'time_100_unr': 2.41024,
 'dor_100_unr': 0.9935579822980823,
 'dor_100_plus': nan,
 'dor_100_min': 0.9451646599874503,
 'error': None}

## Save everything

In [ ]:
df_out = pd.DataFrame([r for r in results if r.get('error') is None])
df_out = df_out.drop(columns=['error'], errors='ignore')
df_out.head()

,ID_INSPIRE,logAge,[M/H],SNR,mass_frac,time_75,time_90,time_100,dor_100,univ_age,...,time_90_reg,time_100_reg,dor_100_reg,mass_frac_unr,time_75_unr,time_90_unr,time_100_unr,dor_100_unr,dor_100_plus,dor_100_min
0,J1317+4336,10.022639,0.331675,14.840348,0.99983,1.00836,1.85064,2.48730,0.778706,11.201004,...,1.85064,2.48730,1.174649,1.00000,1.00836,1.43734,2.41024,0.993558,NaN,0.945165
1,J1346+6021,10.042375,0.336479,23.601007,0.89175,0.97059,3.09954,9.78538,0.543018,11.680153,...,3.09954,9.78538,0.903112,0.99883,0.97059,1.39903,2.44768,1.001474,NaN,0.957757
2,J0807+5317,10.039890,0.341108,21.298576,0.94550,1.86999,2.57014,10.28903,0.460724,11.543202,...,2.57014,10.28903,0.673844,0.99895,1.07707,1.49518,2.47666,0.982872,NaN,0.924830
3,J1558+2707,10.019037,0.258668,18.112293,0.90461,1.78423,2.82776,9.46468,0.473567,11.469951,...,2.82776,9.46468,0.686557,0.98806,1.11153,1.60727,3.87594,0.933323,NaN,0.905862
4,J1647+2956,10.015589,0.255521,19.235961,0.95915,1.19495,1.74517,8.60648,0.559336,11.301923,...,1.35270,8.60648,0.973449,0.97756,1.19495,1.74517,5.13534,0.880537,NaN,0.712027


In [ ]:
df_out.to_csv('einspire_refitting_results/output.csv', index=False)

In [ ]:
print(df_out)

     ID_INSPIRE     logAge     [M/H]        SNR  mass_frac  time_75  time_90  \
0    J1317+4336  10.022639  0.331675  14.840348    0.99983  1.00836  1.85064   
1    J1346+6021  10.042375  0.336479  23.601007    0.89175  0.97059  3.09954   
2    J0807+5317  10.039890  0.341108  21.298576    0.94550  1.86999  2.57014   
3    J1558+2707  10.019037  0.258668  18.112293    0.90461  1.78423  2.82776   
4    J1647+2956  10.015589  0.255521  19.235961    0.95915  1.19495  1.74517   
..          ...        ...       ...        ...        ...      ...      ...   
425  J1001+2340   9.675251  0.279928  18.691985    0.64130  8.00408  8.97785   
426  J1502+2306   9.952603  0.038899  21.962870    0.83357  2.11551  4.40159   
427  J0815+0635   9.684465 -0.318986  22.774130    0.13622  6.60983  7.77528   
428  J1313+6540   9.993735  0.327307  31.487003    0.91805  1.34749  2.39156   
429  J0103+1426   9.945279  0.124701  23.466670    0.64230  3.58867  4.94980   

     time_100   dor_100   univ_age  ...

In [ ]:
tie_balmer = True
limit_doublets = True

c = 299792.458  # speed of light in km/s

regul_err = 0.1 # Large regularization error

vel = 0   # eq.(8) of Cappellari (2017)

moments = [4, 2, 2]

gas_reddening = 0 if tie_balmer else None

logAges = []
metals = []
snrs = []